# LangChain RAG 튜토리얼

이 노트북은 LangChain의 최신 Agent API를 사용하여 RAG 파이프라인을 구축합니다.

**3가지 방식:**
1. **에이전트 + 검색 도구** - `create_agent` + `@tool`로 검색 도구를 에이전트에 연결
2. **동적 프롬프트 (RAG Chain)** - `@dynamic_prompt`로 검색 결과를 시스템 프롬프트에 주입
3. **소스 문서 반환** - `AgentMiddleware`로 소스 문서를 함께 반환

**파이프라인:**
```
색인화: 문서 로드 → 분할 → 임베딩 → 벡터 저장소
검색: 사용자 쿼리 → 벡터 유사성 검색
생성: 검색 결과 + 질문 → LLM → 답변
```

## 0단계: 환경 설정

In [9]:
import os
from typing import Any
import bs4
from dotenv import load_dotenv
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import dynamic_prompt, ModelRequest, AgentMiddleware

load_dotenv()

True

## 1단계: 문서 로드 및 색인화

Lilian Weng의 블로그 포스트를 로드하고, 청크로 분할한 뒤 벡터 저장소에 색인화합니다.

`bs4.SoupStrainer`로 블로그의 본문 부분(`post-title`, `post-header`, `post-content`)만 파싱합니다.

In [10]:
# BeautifulSoup으로 필요한 부분만 파싱
bs4_strainer = bs4.SoupStrainer(
    class_=("post-title", "post-header", "post-content")
)
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

# 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(docs)

# 벡터 저장소 생성 및 색인화
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = InMemoryVectorStore(embeddings)
document_ids = vector_store.add_documents(documents=all_splits)

print(f"로드된 문서 수: {len(docs)}")
print(f"분할 후 청크 수: {len(all_splits)}")
print(f"색인된 문서 ID 수: {len(document_ids)}")

로드된 문서 수: 1
분할 후 청크 수: 63
색인된 문서 ID 수: 63


## 2단계: 방법 1 - 에이전트 + 검색 도구

`@tool(response_format="content_and_artifact")`로 검색 도구를 정의하고,
`create_agent`로 에이전트에 연결합니다.

에이전트가 **스스로 검색이 필요한지 판단**하고, 필요하면 도구를 호출합니다.

In [11]:
# 검색 도구 정의
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs


# 모델 및 에이전트 생성
model = init_chat_model("gpt-4.1", temperature=0)

tools = [retrieve_context]
# 블로그 포스트에서 컨텍스트를 검색하는 도구에 접근할 수 있습니다.
# 사용자 쿼리에 답변하기 위해 도구를 사용하세요.
prompt = (
    "You have access to a tool that retrieves context from a blog post. "
    "Use the tool to help answer user queries."
)
agent = create_agent(model, tools, system_prompt=prompt)

print("에이전트 생성 완료")

에이전트 생성 완료


In [12]:
# 멀티 스텝 쿼리 실행 (Task Decomposition에 대해 질문)
query = (
    "What is the standard method for Task Decomposition?\n\n"  # 작업 분해의 표준 방법은 무엇인가?
    "Once you get the answer, look up common extensions of that method."  # 답을 얻으면, 그 방법의 일반적인 확장을 찾아봐.
)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What is the standard method for Task Decomposition?

Once you get the answer, look up common extensions of that method.
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (call_tI5eRbhDDQvEmoOmaFq1kofy)
 Call ID: call_tI5eRbhDDQvEmoOmaFq1kofy
  Args:
    query: standard method for Task Decomposition
================================= Tool Message =================================
Name: retrieve_context

Source: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'start_index': 2578}
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Another quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external cla

## 3단계: 방법 2 - 동적 프롬프트 (RAG Chain)

`@dynamic_prompt` 데코레이터를 사용하여 검색 결과를 시스템 프롬프트에 자동으로 주입합니다.

도구 호출 없이 **모든 쿼리에 대해 자동으로 검색**하는 방식입니다.

In [13]:
model2 = init_chat_model("gpt-4.1", temperature=0)


@dynamic_prompt
def prompt_with_context(request: ModelRequest) -> str:
    """Inject context into state messages."""
    last_query = request.state["messages"][-1].text
    retrieved_docs = vector_store.similarity_search(last_query)
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
    # 당신은 유용한 어시스턴트입니다. 응답에 다음 컨텍스트를 사용하세요.
    system_message = (
        "You are a helpful assistant. Use the following context in your response:"
        f"\n\n{docs_content}"
    )
    return system_message


rag_chain = create_agent(model2, tools=[], middleware=[prompt_with_context])

print("RAG Chain 생성 완료")

RAG Chain 생성 완료


In [14]:
# RAG Chain 실행 (Task Decomposition이란?)
query = "What is task decomposition?"  # Task Decomposition이란 무엇인가?

for step in rag_chain.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is task decomposition?
================================== Ai Message ==================================

Task decomposition is the process of breaking down a complex task into smaller, more manageable sub-tasks or steps. This helps an agent (such as an AI system or a human) to plan and execute the overall task more effectively by focusing on one step at a time.

In the context of AI and large language models (LLMs), task decomposition can be achieved in several ways:

1. **Simple Prompting:** Asking the model directly for steps or subgoals, such as "Steps for XYZ" or "What are the subgoals for achieving XYZ?".
2. **Task-Specific Instructions:** Using prompts tailored to the specific task, like "Write a story outline" for novel writing.
3. **Human Inputs:** Having a human provide the breakdown of the task.

There are also more advanced methods, such as:
- **Chain of Thought (CoT):** Instructing the mo

## 4단계: 방법 3 - 소스 문서 반환 (Middleware)

`AgentMiddleware`를 상속하여 검색된 소스 문서를 상태(state)에 함께 저장합니다.

답변과 함께 **어떤 문서를 참조했는지** 확인할 수 있습니다.

In [15]:
model3 = init_chat_model("gpt-4.1", temperature=0)


class State(AgentState):
    context: list[Document]


class RetrieveDocumentsMiddleware(AgentMiddleware[State]):
    state_schema = State

    def before_model(self, state: AgentState) -> dict[str, Any] | None:
        last_message = state["messages"][-1]
        retrieved_docs = vector_store.similarity_search(last_message.text)
        docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
        augmented_message_content = (
            f"{last_message.text}\n\n"
            "Use the following context to answer the query:\n"
            f"{docs_content}"
        )
        return {
            "messages": [last_message.model_copy(update={"content": augmented_message_content})],
            "context": retrieved_docs,
        }


agent_with_sources = create_agent(
    model3,
    tools=[],
    middleware=[RetrieveDocumentsMiddleware()],
)

print("소스 반환 에이전트 생성 완료")

소스 반환 에이전트 생성 완료


In [16]:
# 실행 및 소스 문서 확인
query = "What is task decomposition?"  # 작업 분해란 무엇인가?

result = agent_with_sources.invoke(
    {"messages": [{"role": "user", "content": query}]}
)

# 최종 답변
print("답변:")
result["messages"][-1].pretty_print()

# 소스 문서 출력
print(f"\n참조한 소스 문서 수: {len(result['context'])}")
for i, doc in enumerate(result["context"]):
    print(f"\n소스 {i + 1}:")
    print(f"  시작 인덱스: {doc.metadata.get('start_index', 'N/A')}")
    print(f"  내용 미리보기: {doc.page_content[:100]}...")

답변:
================================== Ai Message ==================================

**Task decomposition** is the process of breaking down a complex task into smaller, more manageable sub-tasks or steps. This makes it easier for an agent (such as a large language model, or LLM) to plan, execute, and track progress toward completing the overall goal.

Based on the provided context, task decomposition can be achieved in several ways:

1. **LLM with Simple Prompting:**  
   By asking the LLM directly for steps or subgoals (e.g., "Steps for XYZ.\n1." or "What are the subgoals for achieving XYZ?"), the model generates a list of smaller tasks needed to accomplish the main objective.

2. **Task-Specific Instructions:**  
   Using prompts tailored to the domain (e.g., "Write a story outline" for novel writing), the LLM produces a structured breakdown relevant to the specific task.

3. **Human Inputs:**  
   Humans can manually specify how to break down a task into sub-tasks.

4. **LLM+P (LLM